In [19]:
import random
random.seed(42)
import torch
torch.manual_seed(42)
words = open("names.txt","r").read().splitlines()
random.shuffle(words)
n = len(words)
n1 = int(0.8 * n)
n2 = int(0.9 * n)
train = words[:n1]
dev = words[n1:n2]
test = words[n2:]

In [20]:
chars = sorted(list(set("".join(words))))
stoi = {s:(i+1) for i,s in enumerate(chars)}
stoi["."] = 0
itos = {i:s for s,i in stoi.items()}


In [21]:
#BIGRAM
import torch
def build_dataset_bigram(words):
    xs, ys = [], []
    for w in words:
        chs = ["."] + list(w) + ["."]
        for ch1, ch2 in zip(chs, chs[1:]):
            xs.append(stoi[ch1])
            ys.append(stoi[ch2])
    xs = torch.tensor(xs)
    ys = torch.tensor(ys)
    return xs, ys
    
xs, ys = build_dataset_bigram(train)
dev_xs, dev_ys = build_dataset_bigram(dev)
test_xs, test_ys = build_dataset_bigram(test)

In [22]:
# gradient descent (train set)
import torch.nn.functional as F

W = torch.randn((27, 27), requires_grad=True)
num = xs.nelement()

for k in range(100):
  
    # forward pass
    logits = W[xs] 
    counts = logits.exp() 
    probs = counts / counts.sum(1, keepdims=True) 
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01*(W**2).mean()
  
    #backward pass
    W.grad = None 
    loss.backward()
    W.data += -50 * W.grad
print(loss.item())

2.490511655807495


In [23]:
#dev set

num = dev_xs.nelement()

logits = W[dev_xs]                                 
counts = logits.exp()
probs = counts / counts.sum(1, keepdims=True)      
loss = -probs[torch.arange(num), dev_ys].log().mean()

print(loss.item())

2.4727306365966797


In [24]:
# test set


num = test_xs.nelement()

logits = W[test_xs]                                 
counts = logits.exp()
probs = counts / counts.sum(1, keepdims=True)     
loss = -probs[torch.arange(num), test_ys].log().mean()

print(loss.item())


2.479372978210449


In [25]:
#sampling from the model
for i in range(5):
    ix = 0
    out = []
    while True:
        logits = W[ix]
        counts = logits.exp()
        probs = counts / counts.sum()
        ix = torch.multinomial(probs,num_samples = 1, replacement = True).item()
        out.append(itos[ix])
        if ix == 0:
            break
    print("".join(out))

than.
txavylliyawrlali.
a.
i.
bimielyn.


In [26]:
#TRIGRAM

def build_dataset_trigram(words):
    xs, ys = [], []
    for w in words:
        chs = [".","."] + list(w) + ["."]
        for i in range(len(chs)-2):
            ch1 = chs[i]
            ch2 = chs[i+1]
            ch3 = chs[i+2]
            row = stoi[ch1] * 27 + stoi[ch2]
            col = stoi[ch3]
            xs.append(row)
            ys.append(col)
    xs = torch.tensor(xs)
    ys = torch.tensor(ys)
    return xs, ys
    
xs, ys = build_dataset_trigram(train)
dev_xs, dev_ys = build_dataset_trigram(dev)
test_xs, test_ys = build_dataset_trigram(test)

In [27]:
# gradient descent (train set)
import torch.nn.functional as F

W = torch.randn((729, 27), requires_grad=True)
num = xs.nelement()

for k in range(1000):
  
  # forward pass
    logits = W[xs] 
    counts = logits.exp() 
    probs = counts / counts.sum(1, keepdims=True) 
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01*(W**2).mean()
  
    #backward pass
    W.grad = None 
    loss.backward()
    W.data += -50 * W.grad
print(loss.item()) 


2.2486791610717773


In [28]:
#dev set

num = dev_xs.nelement()

logits = W[dev_xs]                                  
counts = logits.exp()
probs = counts / counts.sum(1, keepdims=True)      
loss = -probs[torch.arange(num),dev_ys].log().mean()


print(loss.item())

2.25596022605896


In [29]:
#test set
num = test_xs.nelement()

logits = W[test_xs]                                 
counts = logits.exp()
probs = counts / counts.sum(1, keepdims=True)     
loss = -probs[torch.arange(num), test_ys].log().mean() 

print(loss.item())

2.255596160888672


In [30]:
#sampling from the model
for i in range(5):
    ix1 = 0
    ix2 = 0
    out = []
    while True:
        row = ix1 * 27 + ix2
        logits = W[row]
        counts = logits.exp()
        probs = counts / counts.sum()
        ix3 = torch.multinomial(probs,num_samples = 1, replacement = True).item()
        
        out.append(itos[ix3])
        if ix3 == 0:
            break
        ix1 = ix2
        ix2 = ix3

    print("".join(out))

kastvzai.
bressonna.
regjor.
tale.
ton.
